# 📗 벡터 진입점에서 관계를 확장하고 근거를 보존합니다

영화 제목을 모르는 사용자가 **줄거리로 기준 영화를 찾고, 그 영화 배우의 다른 출연작까지 요청하는 상황**을 다룹니다.

**실습 질문**

> 가상 현실의 비밀을 알게 된 해커가 주인공인 영화는 무엇인가요? 그 영화와 출연 배우를 한 명 이상 공유하는 다른 작품도 찾아 주세요.

1. **줄거리로 찾기:** 벡터 검색으로 질문과 비슷한 청크와 그 청크가 설명하는 영화를 찾습니다.
2. **관계로 찾기:** 그 영화의 출연 배우를 거쳐 다른 출연작을 찾습니다.

줄거리 유사작만 요청한 질문은 첫 단계에서 마칩니다. 이 실습은 **같은 배우의 다른 작품도 요청했기 때문에** 출연 관계까지 탐색합니다.

## 이번 교안에서 배울 내용

1. `VectorCypherRetriever`로 벡터 검색 뒤 출연 관계 2홉을 확장합니다.
2. 진입 청크 수와 확장 한도를 조절하고, 중복 문서와 근거 경로를 구분합니다.
3. 확장 후보의 원문을 읽고 전문 검색 결과와 같은 문서 ID로 비교합니다.

<img src="images/07_question_actor_paths.png" width="1050" alt="줄거리로 The Matrix를 찾고 공통 배우를 거쳐 다른 출연작을 찾습니다. The Matrix Reloaded는 배우 네 명으로 연결되지만 후보 영화 한 편으로 반환됩니다.">

그림은 배포 데이터에서 배우 4명과 후보 3편만 골라 보여 줍니다. `The Matrix Reloaded`는 공통 배우가 4명이므로 **후보는 한 편, `paths`의 출연 경로는 4개**입니다.


## 준비한 영화 자료를 연결합니다

이전 교안에서 사용한 **Neo4j Movies 그래프 171개 개체·253개 관계, 영화 원문 38편,
원문 청크 571개와 768차원 배포 벡터**를 그대로 사용합니다. 영화 관계는 Neo4j 샘플,
줄거리는 버전이 고정된 Wikipedia 원문입니다. 두 출처를 구분해서 인용합니다.

노트북 폴더에서 시작하며, 패키지·DB 설정은 README를 참고하세요.

검색·그래프·모델에 필요한 라이브러리를 불러옵니다.


In [ ]:
# 검색·DB·모델에 필요한 라이브러리를 불러옵니다.
import os
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field


자료 경로를 지정하고 그래프·벡터 파일을 읽는 함수를 불러옵니다.


In [ ]:
# 노트북 기준 경로입니다. 정답은 상위 폴더를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors


`.env`로 Neo4j에 연결하고 임베딩·답변 모델과 Cypher 실행 함수를 준비합니다.


In [ ]:
# 공통 설정을 읽고 교안의 .env를 우선 적용합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# 모델만 준비합니다. API 호출은 검색·답변 단계에서 합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 준비한 청크를 자동으로 나누지 않습니다.
)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
address = urlparse(neo4j_uri)
print("Neo4j 연결:", address.hostname, address.port or 7687)


def run_cypher(query, **params):
    """Cypher 결과를 행 딕셔너리의 리스트로 반환합니다."""
    # 인자를 Cypher의 $파라미터로 전달합니다.
    with driver.session() as session:
        # 조회 결과를 딕셔너리 목록으로 반환합니다.
        return [record.data() for record in session.run(query, **params)]


영화 그래프·원문·기존 임베딩을 불러오고 자료 규모를 확인합니다.


In [ ]:
# 영화 그래프·원문과 기존 청크 벡터를 불러옵니다.
movies = load_graph(data_dir / "movies_extraction_packet.json")
chunk_ids, vectors = read_vectors(
    data_dir / "embeddings/movies_complete.npz", movies["chunks"], embedding_model
)
# 검색 결과의 원문 ID로 줄거리를 찾을 사전입니다.
documents_by_id = movies["documents"]
print("개체:", len(movies["nodes"]), "관계:", len(movies["relations"]))
print("문서:", len(documents_by_id), "청크:", len(chunk_ids), "벡터:", vectors.shape)


### 영화 그래프와 원문·청크를 적재합니다

원래 ID·속성·관계·출처를 유지하며 `Movie`, `Person`, `Document`, `Chunk`에 저장합니다.
`RAGEntity`는 영화·인물의 공통 ID 조회와 고유 제약을 위해 추가하는 레이블입니다.

ID 중복 방지 제약을 만들고 영화·인물·관계를 Neo4j에 저장합니다.


In [ ]:
# 개체의 standard_id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT rag_entity_id IF NOT EXISTS FOR (n:RAGEntity) REQUIRE n.standard_id IS UNIQUE")
# 원문 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (n:Document) REQUIRE n.id IS UNIQUE")
# 청크 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE")

# 영화·인물을 종류별로 적재합니다.
for entity_type in ["Movie", "Person"]:
    # 현재 종류의 개체만 고릅니다.
    rows = [row for row in movies["nodes"] if row["entity_type"] == entity_type]
    # 현재 종류의 영화·인물 노드를 만들고 속성을 갱신합니다.
    run_cypher(f"""
        // 목록을 개체별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 같은 ID의 노드를 재사용하고, 없으면 생성합니다.
        MERGE (n:RAGEntity {{standard_id: item.standard_id}})
        // 종류 레이블을 추가하고 속성·이름을 갱신합니다.
        SET n:{entity_type}, n += item.properties, n.name = item.name
    """, rows=rows)

# 적재한 노드 사이에 종류별 관계를 연결합니다.
for relation in ["ACTED_IN", "DIRECTED", "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS"]:
    # 현재 유형의 관계만 고릅니다.
    rows = [row for row in movies["relations"] if row["relation"] == relation]
    # 현재 유형의 관계를 연결하고 근거·출처를 저장합니다.
    run_cypher(f"""
        // 목록을 관계별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 관계의 시작·끝 노드를 ID로 찾습니다.
        MATCH (a:RAGEntity {{standard_id: item.subject_id}})
        MATCH (b:RAGEntity {{standard_id: item.object_id}})
        // 같은 유형·방향·근거 ID의 관계가 없으면 만들고, 있으면 재사용합니다.
        MERGE (a)-[r:{relation} {{claim_id: item.claim_id}}]->(b)
        // 관계 속성과 근거·출처를 저장합니다.
        SET r += item.properties, r.evidence = item.evidence,
            r.source_url = item.source_url, r.source_doc_id = item.source_doc_id
    """, rows=rows)


원문·청크·기존 벡터를 영화에 연결하고 전문·벡터 검색 인덱스를 만듭니다.


In [ ]:
# 원문 노드를 먼저 저장합니다.
run_cypher("""
    // 원문 목록을 한 문서당 한 행으로 펼칩니다.
    UNWIND $rows AS item
    // 같은 ID의 원문 노드가 없으면 만들고, 있으면 재사용합니다.
    MERGE (d:Document {id: item.doc_id})
    // 제목·본문·출처 URL을 갱신합니다.
    SET d.title = item.title, d.text = item.text, d.url = item.url
""", rows=list(documents_by_id.values()))

# 원문과 해당 영화를 연결합니다.
run_cypher("""
    // doc_id로 원문을, standard_id로 해당 영화를 찾습니다.
    UNWIND $rows AS item
    MATCH (d:Document {id: item.doc_id})
    MATCH (m:Movie {standard_id: item.standard_id})
    // 원문과 해당 영화를 ABOUT_MOVIE 관계로 연결합니다.
    MERGE (d)-[:ABOUT_MOVIE]->(m)
""", rows=movies["document_links"])

# 청크와 벡터의 순서를 맞춰 전체 저장 목록을 만듭니다.
rows = []
for index, chunk in enumerate(movies["chunks"]):
    # NumPy 벡터를 DB에 전달할 목록으로 바꿉니다.
    rows.append({"id": chunk_ids[index], "doc_id": chunk.metadata["source_doc_id"],
                 "text": chunk.page_content, "embedding": vectors[index].tolist()})

# 전체 청크와 벡터를 한 번에 저장합니다.
run_cypher("""
    UNWIND $rows AS item
    // 원문을 찾고 청크를 ID로 생성·재사용합니다.
    MATCH (d:Document {id: item.doc_id})
    MERGE (c:Chunk {id: item.id})
    // 이 교안의 영화 청크만 검색하도록 전용 레이블을 붙입니다.
    SET c:Day49MovieChunk
    // text는 전문 검색, embedding은 벡터 검색, source_id는 원문별 결과 통합에 씁니다.
    SET c.text = item.text, c.source_id = item.doc_id, c.embedding = item.embedding
    // 청크와 원문을 FROM_DOCUMENT 관계로 연결합니다.
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)

# 이전 버전이 전체 Chunk를 색인했다면 이 교안의 두 인덱스만 다시 만듭니다.
index_properties = {"movie_chunks": ["embedding"], "movie_fulltext": ["text"]}
existing_indexes = run_cypher("""
    SHOW INDEXES YIELD name, labelsOrTypes, properties
    WHERE name IN ['movie_chunks', 'movie_fulltext']
    RETURN name, labelsOrTypes, properties
""")
for index in existing_indexes:
    if index["labelsOrTypes"] != ["Day49MovieChunk"] or index["properties"] != index_properties[index["name"]]:
        run_cypher(f"DROP INDEX `{index['name']}` IF EXISTS")

# 벡터 유사도 검색용 인덱스를 만듭니다.
run_cypher("""
    CREATE VECTOR INDEX movie_chunks IF NOT EXISTS
    // 이 교안의 영화 청크만 벡터 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON c.embedding
    // 저장된 벡터의 768차원과 코사인 유사도를 설정합니다.
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
# 벡터 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_chunks', 300)")
# 단어 일치 검색용 전문 인덱스를 만듭니다.
run_cypher("""
    CREATE FULLTEXT INDEX movie_fulltext IF NOT EXISTS
    // 같은 영화 청크의 text만 전문 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON EACH [c.text]
    // 영어 분석기를 사용하고, 본문 변경을 커밋 시 인덱스에 반영합니다.
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'english', `fulltext.eventually_consistent`: false}}
""")
# 전문 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_fulltext', 300)")
print("기존 문서 임베딩을 적재했습니다. 문서 임베딩 API 호출은 없습니다.")


### 적재된 그래프 전체 구조

청크·원문·영화·인물의 연결과 검색 인덱스를 한눈에 봅니다.

<img src="images/06_loaded_graph_v2.png" width="1000" alt="적재된 영화 그래프: 청크에서 원문과 영화로 연결되며, 인물은 영화 및 다른 인물과 관계를 맺습니다. 검색 인덱스는 청크에 적용됩니다.">


한 영화의 여러 청크가 함께 검색될 수 있습니다. 이번 실습은 영화별로 순위를 합치므로, 검색된 청크 중 영화마다 최고 점수 한 건을 대표로 남깁니다. 청크 자체를 답변 근거로 사용할 때는 같은 영화의 여러 청크를 함께 유지할 수도 있습니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


def show_documents(rows):
    """전달받은 모든 검색 결과를 순서대로 조회해 제목과 본문 앞부분을 보여 줍니다."""
    display(pd.DataFrame(rows))
    # 검색 순위를 함께 전달해 DB 조회 후에도 같은 순서를 유지합니다.
    ranked_ids = [{"rank": rank, "source_id": row["source_id"]}
                  for rank, row in enumerate(rows)]
    # 전달받은 원문 전체의 제목과 본문을 Neo4j에서 한 번에 조회합니다.
    documents = run_cypher("""
        UNWIND $rows AS item
        // 검색 결과의 원문 ID로 저장된 문서를 찾습니다.
        MATCH (d:Document {id: item.source_id})
        RETURN d.id AS source_id, d.title AS title, d.text AS text
        // DB의 반환 순서를 원래 검색 순위에 맞춥니다.
        ORDER BY item.rank
    """, rows=ranked_ids)
    for doc in documents:
        print("\n제목:", doc["title"], "| 원문 ID:", doc["source_id"])
        # 긴 원문은 앞 650자만 출력합니다.
        print(doc["text"][:650])


## 1. 진입 청크와 최종 문서를 구분합니다

**청크**는 영화 원문을 검색하기 좋게 나눈 텍스트 조각입니다.
벡터 검색으로 처음 찾은 청크를 관계 탐색의 시작점으로 사용하므로 **진입 청크**라고 부릅니다.
이 청크가 속한 영화가 **진입 영화**이고, 같은 배우를 따라 추가로 찾은 영화가 **확장 후보**입니다.
`top_k=12`는 최대 청크 12개를 찾는 설정입니다. 같은 영화의 청크가 여러 개 나오면 영화 수는 12편보다 적습니다.
아래 `question`에는 전체 요청 중 **기준 영화를 찾을 줄거리 부분**을 넣습니다. 같은 배우의 다른 작품을 찾는 조건은 다음 절의 Cypher로 처리합니다.

질문 벡터는 한 번 만들고 이후 비교에서 재사용합니다.

한글 질문을 임베딩하고 유사한 청크 12개를 조회합니다.


In [ ]:
question = "해커가 자신이 사는 세상이 기계가 통제하는 가상 현실임을 알게 되는 영화"
# 질문 벡터를 한 번 만들어 재사용합니다.
query_vector = embedding_model.embed_query(question)
raw_result = vector_retriever.search(query_vector=query_vector, top_k=12)
display(pd.DataFrame([item.metadata for item in raw_result.items]))


검색 청크를 원문별로 합쳐 고유 영화 수와 상위 작품을 확인합니다.


In [ ]:
# 찾은 청크 수와 중복을 뺀 영화 수를 비교합니다.
seed_documents = vector_documents(query_vector, top_k=12)
print("진입 청크 수:", len(raw_result.items), "고유 문서 수:", len(seed_documents))
show_documents(seed_documents)


### 🖐️ 함께 따라하기: 조난 줄거리의 진입점을 찾습니다

질문을 `비행기 추락 사고에서 살아남은 남자가 외딴 무인도에서 홀로 생존하는 영화`로 바꾸세요. `practice_vector`에 질문 벡터를 담고, 최대 청크 12개를 검색해 합친 문서 목록을 `practice_seeds`에 담으세요. 고유 문서 수와 검색된 모든 영화의 제목·본문 앞부분을 확인합니다. 결과는 이후 따라하기에서도 사용합니다.

무인도 생존 질문과 비슷한 영화를 벡터 검색으로 찾고 상위 원문을 읽습니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

청크 12개를 받았는데 영화가 4편이면 검색기가 실패한 것인가요?

<details><summary>해설 보기</summary>

그렇지 않습니다. 한 영화의 여러 청크가 검색될 수 있습니다. 검색한 청크 수와 중복을 뺀 영화 수를 구분해야 합니다.

</details>


## 2. VectorCypherRetriever로 다른 영화까지 확장합니다

이제 질문의 “**그 영화와 출연 배우를 공유하는 다른 작품**”을 찾습니다. 앞에서 찾은 영화의 출연 배우를 확인하고, 그 배우가 출연한 다른 영화로 이동합니다.

확장 후보의 선정 기준은 **공통 출연 배우**입니다. 확장 후보의 줄거리가 처음 질문과 비슷할 필요는 없습니다.

<img src="images/01_vector_graph.png" width="1050" alt="청크에서 원문과 영화로 이동한 뒤 배우의 출연 관계 두 개로 다른 영화까지 확장하는 상세 구조">

`VectorCypherRetriever`는 먼저 질문과 비슷한 청크를 찾고, 그 결과로 `retrieval_query`를 실행합니다.

| 변수 | 담긴 값 |
|---|---|
| `node` | 벡터 검색으로 찾은 `Chunk` 노드 |
| `score` | 질문 벡터와 해당 청크 벡터의 유사도 점수. 클수록 더 비슷합니다. |

추가 Cypher는 청크가 속한 영화를 찾은 뒤, 같은 배우가 출연한 다른 영화까지 조회합니다.
이때 `score`는 처음 찾은 청크의 점수로 유지됩니다.

```text
청크 -> 원문 -> 영화 A <- 배우 -> 영화 B <- 원문
```

`ACTED_IN`은 **배우에서 영화로** 향합니다. 영화 A에서 B까지는 2홉이며, 청크에서 출발하면 원문 연결 관계도 추가됩니다.
줄거리가 비슷한 후보를 찾는 것과 배우가 겹치는 후보를 찾는 것은 서로 다른 기준입니다.

[VectorCypherRetriever 공식 문서](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_rag.html#vector-cypher-retriever-user-guide)


### 확장 Cypher를 작성합니다

`(seed:Movie)`에서 `Movie`는 DB의 레이블, `seed`는 찾은 노드를 가리키는 **쿼리 변수**입니다.

- `seed`: 청크에서 `FROM_DOCUMENT → ABOUT_MOVIE`를 따라 찾은 기준 영화
- `other`: 기준 영화와 같은 배우가 출연한 다른 영화
- `min_year`: 다른 영화의 최소 개봉연도
- `max_per_seed`: 청크 하나에서 추가로 찾을 영화 수의 상한

**`paths`는 기준 영화와 후보 영화가 어떤 배우를 통해 연결되는지 기록한 목록**입니다.

| 항목 | 의미 |
|---|---|
| `seed_title` | 기준 영화 제목 |
| `actor` | 두 영화에 공통으로 출연한 배우 |
| `first_claim` | 배우가 기준 영화에 출연한 관계의 근거 ID |
| `second_claim` | 배우가 후보 영화에 출연한 관계의 근거 ID |

`collect`는 출연 경로를 목록으로 모으고, `DISTINCT`는 같은 항목의 중복을 제거합니다.

공통 배우가 여러 명이면 `paths`에 여러 항목이 들어갑니다. 이 목록은 **후보 영화를 찾은 이유를 설명하고 출연 근거를 확인할 때** 사용합니다.

`CALL (node) { ... }`는 검색된 청크마다 내부 쿼리를 실행합니다. 후보 영화별로 `paths`를 만들고, 원문 ID 오름차순으로 `max_per_seed`편까지 반환합니다.

처음 찾은 청크에서 출발해 같은 배우의 다른 영화와 원문을 찾는 Cypher를 작성합니다.


In [ ]:
# 검색된 청크마다 공통 배우의 다른 영화로 확장할 쿼리입니다.
expansion_query = """
// 검색된 청크마다 아래 관계 확장 쿼리를 실행합니다.
CALL (node) {
    // 청크의 원문이 설명하는 영화를 찾습니다.
    MATCH (node)-[:FROM_DOCUMENT]->(:Document)-[:ABOUT_MOVIE]->(seed:Movie)
    // 공통 출연 배우를 거쳐 다른 영화와 원문을 찾습니다.
    MATCH (seed)<-[first:ACTED_IN]-(actor:Person)-[second:ACTED_IN]->(other:Movie)
    MATCH (other_doc:Document)-[:ABOUT_MOVIE]->(other)
    // 기준 영화를 제외하고 최소 개봉연도를 적용합니다.
    WHERE other <> seed AND other.released >= $min_year
    // 후보 영화별로 배우와 양쪽 출연 관계 ID를 모읍니다.
    RETURN other_doc.id AS source_id, other_doc.title AS title,
           collect(DISTINCT {seed_title: seed.title, actor: actor.name,
               first_claim: first.claim_id, second_claim: second.claim_id}) AS paths
    // 이 청크의 후보를 원문 ID순으로 정렬해 최대 max_per_seed편까지 반환합니다.
    ORDER BY source_id
    LIMIT $max_per_seed
}
// 시작 청크의 ID·유사도와 확장한 영화의 원문·경로를 반환합니다.
RETURN node.id AS seed_chunk_id, score AS seed_score, source_id, title, paths
"""


벡터 검색 뒤 배우를 따라 다른 영화를 찾는 검색기를 만들고 중복 결과를 합칩니다.


In [ ]:
def format_expansion(record):
    """후보 영화의 제목과 그 영화를 찾은 청크·출연 경로를 반환합니다."""
    row = record.data()
    return RetrieverResultItem(content=row["title"], metadata=row)


# 벡터 검색 뒤 관계 확장 쿼리를 실행하는 검색기를 만듭니다.
graph_vector_retriever = VectorCypherRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    retrieval_query=expansion_query, result_formatter=format_expansion,
)


def merge_expansions(result):
    """중복 문서를 합치되 서로 다른 출연 경로는 모두 남깁니다.

    Args:
        result: VectorCypherRetriever.search의 반환값.
    Returns:
        원문 ID별로 출발한 청크 ID와 중복 없는 출연 경로를 담은 목록.
    """
    merged = {}
    for item in result.items:
        row = item.metadata
        source_id = row["source_id"]
        # 같은 원문 ID의 결과를 한곳에 모아 출발 청크와 출연 경로를 기록합니다.
        target = merged.setdefault(source_id, {
            "source_id": source_id, "title": row["title"],
            "seed_score": row["seed_score"], "seed_chunks": [], "paths": [],
        })
        # 같은 후보를 찾은 청크가 여럿이면 가장 높은 유사도를 남깁니다.
        target["seed_score"] = max(target["seed_score"], row["seed_score"])
        # 같은 후보를 찾은 서로 다른 시작 청크를 모두 기록합니다.
        if row["seed_chunk_id"] not in target["seed_chunks"]:
            target["seed_chunks"].append(row["seed_chunk_id"])
        # 같은 경로는 한 번만 남기고, 다른 배우의 출연 경로는 보존합니다.
        for path in row["paths"]:
            if path not in target["paths"]:
                target["paths"].append(path)
    # seed_score가 큰 후보부터 정렬하고, 동점이면 원문 ID순으로 정렬합니다.
    return sorted(merged.values(), key=lambda row: (-row["seed_score"], row["source_id"]))


벡터 검색으로 찾은 청크 3개에서 배우를 따라 1990년 이후 영화를 최대 10편씩 찾습니다.


In [ ]:
# 청크 3개를 먼저 찾고, 각 청크에서 다른 영화를 최대 10편씩 찾습니다.
expanded_result = graph_vector_retriever.search(
    query_vector=query_vector, top_k=3,
    query_params={"min_year": 1990, "max_per_seed": 10},
)
expanded_documents = merge_expansions(expanded_result)
print("확장 반환 행:", len(expanded_result.items), "고유 문서:", len(expanded_documents))
show_documents(expanded_documents)


여러 청크에서 같은 후보 영화를 찾으면, `seed_score`에는 **그 청크들의 유사도 중 가장 높은 값**을 남깁니다.
후보 영화 자체의 줄거리 유사도를 새로 계산한 값은 아닙니다.
`seed_chunks`에는 출발한 청크 ID, `paths`에는 후보로 이어지는 배우와 두 출연 관계 ID가 담깁니다.


### 🖐️ 함께 따라하기: 조난 이야기에서 출연 관계를 확장합니다

`practice_vector`를 재사용해 청크 3개에서 시작하고, 1995년 이후의 다른 영화를 청크당 최대 8편 찾으세요. `practice_expanded`에 중복을 합친 결과를 담으세요. 한 후보의 `seed_chunks`·`paths`를 읽고, 어느 진입 영화와 어느 배우로 연결됐는지 설명하세요.

무인도 생존 질문으로 찾은 영화와 배우가 겹치는 1995년 이후 영화를 찾습니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

배우가 3명 겹치는 영화의 seed_score를 세 번 더해도 될까요?

<details><summary>해설 보기</summary>

배우가 많이 겹친다는 이유로 줄거리 유사도가 커지는 것은 아닙니다. 같은 후보를 찾은 청크 점수 중 최댓값만 남기고, 출연 경로는 근거로 보관합니다.

</details>


## 3. Neo4j 전문 검색을 같은 원문 ID로 연결합니다

<img src="images/08_fulltext_language_search_v2.png" width="1000">

- **언어 처리:** Jev가 언어를 판단합니다. 한글은 구조화된 영어 단어 목록으로 바꿔 `OR`로 연결하고, 영어는 그대로 검색합니다.
- **전문 검색:** `db.index.fulltext.queryNodes`로 청크와 관련도 점수(`score`)를 받습니다. Neo4j는 Lucene을 사용하며, Lucene의 기본 점수 모형은 **BM25**입니다.
- **결과 연결:** 같은 원문에서는 최고 점수 청크를 남기고, `source_id`로 벡터 검색 결과와 연결합니다.

전문 검색은 `queryNodes` 프로시저를 사용하며, 영화 원문에 맞춰 `english` 분석기를 사용합니다.


### 인덱스 상태와 설정을 확인합니다

`ONLINE`과 `english`를 확인하세요. `CREATE ... IF NOT EXISTS`는 기존 인덱스의 분석기를 바꾸지 않습니다.
다른 설정으로 비교하려면 별도 이름의 인덱스를 만들고 검색 대상을 바꿉니다.

전문 인덱스의 준비 상태와 영어 분석기 설정을 확인합니다.


In [ ]:
# 인덱스 상태와 실제 분석기 설정을 확인합니다.
display(pd.DataFrame(run_cypher("""
    SHOW FULLTEXT INDEXES YIELD name, state, populationPercent, labelsOrTypes, options
    WHERE name = 'movie_fulltext'
    RETURN name, state, populationPercent, labelsOrTypes, options
""")))


언어에 맞는 검색어로 전문 검색하고, 원문별 최고 점수 청크를 반환하는 함수를 만듭니다.


In [ ]:
# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows


전문·벡터 검색은 줄거리로 기준 영화를 찾고, 관계 확장은 같은 배우의 다른 작품을 찾습니다. 각 단계가 가져온 문서 ID를 비교합니다.


In [ ]:
# 한글 질문을 영어 검색어로 바꿔 전문 검색합니다.
fulltext_rows = fulltext_documents(question, top_k=12)
route_rows = []
for route, rows in [("전문 검색", fulltext_rows), ("Dense", seed_documents), ("관계 확장", expanded_documents)]:
    for order, row in enumerate(rows, start=1):
        route_rows.append({"route": route, "display_order": order,
                           "source_id": row["source_id"], "title": documents_by_id[row["source_id"]]["title"]})
display(pd.DataFrame(route_rows))


### 🖐️ 함께 따라하기: 한글 질문과 고유 문서 ID를 확인합니다

`비행기 추락 뒤 무인도에서 생존하는 이야기`를 전문 검색에 넣고 최대 청크 12개를 찾으세요. 결과를 `practice_fulltext`에 담아 `practice_seeds`와 겹치는 원문 ID를 출력하세요. 한쪽에서만 찾은 원문도 읽습니다.

무인도 생존 질문의 전문·벡터 검색 결과에서 겹치는 원문을 확인합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

그래프 확장 목록의 두 번째 영화는 전문 검색 2위와 같은 의미의 2위인가요?

<details><summary>해설 보기</summary>

아닙니다. 여기서는 진입 청크의 점수와 ID로 출력 순서를 정했습니다. 대상 영화 자체의 관련성 순위가 아닙니다. 다음 교안에서는 순위가 없는 관계 조건 집합을 따로 처리합니다.

</details>


## 이번 강의 정리

- `top_k`는 진입 청크 수이고, 확장 한도는 진입 청크별 대상 영화 수입니다.
- 줄거리로 기준 영화를 찾은 뒤, 같은 배우의 다른 출연작을 요청한 조건을 Cypher로 처리합니다.
- 같은 문서의 중복은 합치고, 서로 다른 출연 경로와 원문 ID는 보존합니다.
- 다음 교안에서는 같은 데이터로 세 검색 경로를 통합하고, 관계 조건을 지킨 후보를 재정렬해 답변합니다.


## 교안 01 핵심 코드 이어서 보기

본문에서 그래프와 인덱스를 적재한 뒤 실행합니다.


DB·임베딩·검색어 변환 모델을 연결합니다.


In [ ]:
# DB·벡터 검색·언어 판단·구조화 출력에 필요한 라이브러리를 불러옵니다.
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field

# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

# 한글 질문을 영어 검색어로 바꿀 모델을 준비합니다.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def run_cypher(query, **params):
    """Cypher의 조회 결과를 사전 목록으로 반환합니다."""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


벡터 검색으로 영화별 대표 청크를 찾는 함수를 만듭니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


언어를 판단하고 전문 검색 결과를 원문별로 합치는 함수를 만듭니다.


In [ ]:
# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows


처음 찾은 영화에서 같은 배우의 다른 영화로 확장할 Cypher를 작성합니다.


In [ ]:
# 검색된 청크마다 공통 배우의 다른 영화로 확장할 쿼리입니다.
expansion_query = """
// 검색된 청크마다 아래 관계 확장 쿼리를 실행합니다.
CALL (node) {
    // 청크의 원문이 설명하는 영화를 찾습니다.
    MATCH (node)-[:FROM_DOCUMENT]->(:Document)-[:ABOUT_MOVIE]->(seed:Movie)
    // 공통 출연 배우를 거쳐 다른 영화와 원문을 찾습니다.
    MATCH (seed)<-[first:ACTED_IN]-(actor:Person)-[second:ACTED_IN]->(other:Movie)
    MATCH (other_doc:Document)-[:ABOUT_MOVIE]->(other)
    // 기준 영화를 제외하고 최소 개봉연도를 적용합니다.
    WHERE other <> seed AND other.released >= $min_year
    // 후보 영화별로 배우와 양쪽 출연 관계 ID를 모읍니다.
    RETURN other_doc.id AS source_id, other_doc.title AS title,
           collect(DISTINCT {seed_title: seed.title, actor: actor.name,
               first_claim: first.claim_id, second_claim: second.claim_id}) AS paths
    // 이 청크의 후보를 원문 ID순으로 정렬해 최대 max_per_seed편까지 반환합니다.
    ORDER BY source_id
    LIMIT $max_per_seed
}
// 시작 청크의 ID·유사도와 확장한 영화의 원문·경로를 반환합니다.
RETURN node.id AS seed_chunk_id, score AS seed_score, source_id, title, paths
"""


관계 확장 검색기와 중복 문서를 합치는 함수를 만듭니다.


In [ ]:
def format_expansion(record):
    """후보 영화의 제목과 그 영화를 찾은 청크·출연 경로를 반환합니다."""
    row = record.data()
    return RetrieverResultItem(content=row["title"], metadata=row)


# 벡터 검색 뒤 관계 확장 쿼리를 실행하는 검색기를 만듭니다.
graph_vector_retriever = VectorCypherRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    retrieval_query=expansion_query, result_formatter=format_expansion,
)


def merge_expansions(result):
    """중복 문서를 합치되 서로 다른 출연 경로는 모두 남깁니다.

    Args:
        result: VectorCypherRetriever.search의 반환값.
    Returns:
        원문 ID별로 출발한 청크 ID와 중복 없는 출연 경로를 담은 목록.
    """
    merged = {}
    for item in result.items:
        row = item.metadata
        source_id = row["source_id"]
        # 같은 원문 ID의 결과를 한곳에 모아 출발 청크와 출연 경로를 기록합니다.
        target = merged.setdefault(source_id, {
            "source_id": source_id, "title": row["title"],
            "seed_score": row["seed_score"], "seed_chunks": [], "paths": [],
        })
        # 같은 후보를 찾은 청크가 여럿이면 가장 높은 유사도를 남깁니다.
        target["seed_score"] = max(target["seed_score"], row["seed_score"])
        # 같은 후보를 찾은 서로 다른 시작 청크를 모두 기록합니다.
        if row["seed_chunk_id"] not in target["seed_chunks"]:
            target["seed_chunks"].append(row["seed_chunk_id"])
        # 같은 경로는 한 번만 남기고, 다른 배우의 출연 경로는 보존합니다.
        for path in row["paths"]:
            if path not in target["paths"]:
                target["paths"].append(path)
    # seed_score가 큰 후보부터 정렬하고, 동점이면 원문 ID순으로 정렬합니다.
    return sorted(merged.values(), key=lambda row: (-row["seed_score"], row["source_id"]))


같은 질문으로 벡터·전문 검색을 실행하고 출연 관계로 확장합니다.


In [ ]:
question = "해커가 자신이 사는 세상이 기계가 통제하는 가상 현실임을 알게 되는 영화"
# 질문을 한 번 임베딩하고 청크 3개에서 다른 영화를 최대 10편씩 찾습니다.
query_vector = embedding_model.embed_query(question)
# 두 검색 결과는 원문 ID로 비교할 수 있습니다.
seed_documents = vector_documents(query_vector, top_k=12)
fulltext_rows = fulltext_documents(question, top_k=12)
result = graph_vector_retriever.search(
    query_vector=query_vector, top_k=3,
    query_params={"min_year": 1990, "max_per_seed": 10},
)
expanded_documents = merge_expansions(result)
{"vector": seed_documents, "fulltext": fulltext_rows, "graph": expanded_documents}
